**PLEASE DO NOT DISTRIBUTE!**

        ⚠️ The normalization layer used in the language model is RMSNorm which differs from the regular LayerNorm.

        ⚠️ The feedforward block used in the language model of this colab differs from a regular MLP block. Here we have 3 lnear layers insead of 2 linear layers. See Fig. 5 in https://docs.nvidia.com/deeplearning/transformer-engine/user-guide/examples/te_llama/tutorial_accelerate_hf_llama_with_te.html.

# **Background**

The Challenge consists of different challenges including:

*   Identifying bugs, and getting the code working. This is designed to test your ability to grapple with real world engineering challenges.
*   Testing your ability to generate code for a specified problem.
*   An opportunity for you to attempt an optional challenge question that extends the original problem set.

**This challenge is motivated by the process of me debugging SmolLM code. This notebook, specifically, will give a very very through comments to help you understand each component and each bug hopefully by just reading through it.** Ideally, by reading through this notebook you can understand how I reproduced SmolLM architecture including the implementation details.

## **Coding Challenge Part 1: Debugging custom SmolLM code [10 points]**

This challenge is to let you try to debug and fix a bare-bones implementation of the following model.

**Model** : SmolLM-135M can be found at [HuggingFace](https://huggingface.co/HuggingFaceTB/SmolLM-135M).

We have 10 bugs in the following implementation.
There is a `check_solution` function for your convenience to verify you have correctly identified all the bugs. If you have found all bugs, the generated outputs will match the reference model exactly.

**Rules**:
1. **Bug Definition:**
  - There are 10 bugs to be fixed.
  - A bug is *defined as **{incorrect, missing, unnecessary}** lines of code*.
  - You earn 1 point for each correctly identified and fixed bug.
2. **Fix Guidelines:**
  - You are encouraged to make the smallest possible fix, wherever possible (e.g. edit a line instead of replacing it entirely).
  - Do not optimize the code; only fix the bugs. The implementation is *intentionally* non-optimized but valid.
3. **Documentation:** Document each fix by adding a comment on the line above the fix: : `### BUG FIX ###`.
4. **Sections:** *1. Setup [Helper Functions]* and *3. Test* don't contain bugs and shouldn't be changed.
5. **Submission:** Your final submission should be the exact same file except with your proposed fixes and the respective comments as per Rule #3.

## 1. Setup [Helper Functions]

In [ ]:
######################################################################################################################
############################################## DO NOT CHANGE[START] ##################################################
######################################################################################################################


# [Don't use. Rate limit issues.] Use gdown to get weights file(BareBones_SmolLM-135M.pt) at https://drive.google.com/file/d/1tY46FSJEhGYRrfKRQTjJ1Cc7q9psaKUU/view . gdown should be installed by default else use `pip install gdown`
# !gdown 1tY46FSJEhGYRrfKRQTjJ1Cc7q9psaKUU


# [Recommended]Use HF to download the weights
!git lfs install
!git clone https://huggingface.co/dsouzadaniel/C4AI_SMOLLM135
!mv C4AI_SMOLLM135/BareBones_SmolLM-135M.pt ./
!ls

Git LFS initialized.
Cloning into 'C4AI_SMOLLM135'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 6 (delta 0), reused 0 (delta 0), pack-reused 3 (from 1)
Unpacking objects: 100% (6/6), 2.11 KiB | 721.00 KiB/s, done.
BareBones_SmolLM-135M.pt
C4AI_SMOLLM135
Copy_of_SFT_DPO_Practice_Original_20250621.ipynb
Debugging_Journal.ipynb
SFT_DPO_Practice_v4_20250628.ipynb


In [ ]:

# Libraries
import torch
import torch.nn.functional as F
from torch import nn
import math
from transformers import AutoModelForCausalLM, AutoTokenizer

# Model initialization/settings
checkpoint="HuggingFaceTB/SmolLM-135M"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

__reference_model = AutoModelForCausalLM.from_pretrained(checkpoint)
__reference_model.eval()

class smolConfig:
    vocab_size=49152
    hidden_size=576
    intermediate_size=1536 # Size of the intermediate layer in the MLP (feedforward) blocks
    num_hidden_layers = 30
    num_heads = 9
    kv_heads=3
config = smolConfig
# Daniel's Note: The thinking process behind model architecture choices: https://claude.ai/public/artifacts/e7620ea1-14d0-450c-b14a-1c59ae7e9d74


# Helper Functions
def __generate(model, inputs, num_tokens):  # Function to generate text using autoregressive decoding
    collect = [] # List to store generated token IDs
    for _ in range(num_tokens): # Loop to generate specified number of tokens
        output = model(**inputs) # Forward pass: get model predictions for current input

        # print(f"Generation logits shape: {output['logits'].shape}")
        # print(f"Indexing result shape: {output['logits'][0,-1].shape}")

        # output有3个dimensions: batch_size, sequence_length, vocab_size (b, s, h)
        output_id = torch.argmax(output['logits'][0,-1]).item() # Get the most likely next token ID from last position
        # 要每句最后一个token(词)所有hidden dimension里最大logit的,
        # Understand Syntax: output.lotgis[0,-1] gives a vector of vocab_size(49152) dimensions, output.logits[0,-1] = ouput.logits[0,-1,:]. When you want all elements from the remaining dimensions, you can omit the indices.

        # print(f"Token ID: {output_id}, Token: '{tokenizer.decode([output_id])}'")

        collect.append(output_id)
        if output_id==tokenizer.eos_token_id: # Check if we generated an end-of-sequence token
            break # Stop generation if EOS token is produced
        inputs['input_ids'] = torch.unsqueeze(torch.cat([inputs['input_ids'][0],torch.tensor([output_id])]),dim=0) # Append new token to input sequence
        inputs['attention_mask'] = torch.ones_like(inputs['input_ids']) # Update attention mask to include new token
    return tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(collect)) # Convert token IDs back to readable text

def check_solution(prompt, num_tokens, model_A, model_B):
    print()
    print(f"{'>'*20}\n\tPrompt\n{'<'*20}\n{prompt}\n\n")
    model_inputs = tokenizer(prompt, return_tensors='pt')
    print(f"{'>'*30}\n\tModel_A Generation\n{'<'*30}\n{__generate(model_A,  model_inputs, num_tokens)}")
    print("\n\n")
    model_inputs = tokenizer(prompt, return_tensors='pt')
    print(f"{'>'*30}\n\tModel_B Generation\n{'<'*30}\n{__generate(model_B,  model_inputs, num_tokens)}")

######################################################################################################################
############################################### DO NOT CHANGE[END] ###################################################
######################################################################################################################

model.safetensors:   0%|          | 0.00/538M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

## 2. Custom SmolLM (for BugFixes)

RotaryEmbedder Class:

Frequencies are typically set using geometric progression:
$$\theta_i = 10000^{-2i/d_{model}}$$

In [ ]:
## Original + Systematic Debugging Analysis Personal Attempt


def rotate_half(x):
    """
    Rearrange tensor elements for RoPE's specific rotation pattern.

    Purpose: This implements RoPE's unique way of mixing dimensions
    to encode relative position information. It's NOT standard 2D rotation.

    Args:
        x: Input tensor of shape (..., d) where d is even

    Returns:
        Tensor with rearranged elements: [-second_half, first_half]

    Example: For a query vector q[0,0,0,:] = [a, b, c, d] at position 0:
        rotate_half([a, b, c, d]) = [-c, -d, a, b]
    Example 2: [a, b, c, d, e, f] -> [-d, -e, -f, a, b, c]

    This enables the formula q*cos + rotate_half(q)*sin to encode
    relative positions, though not via standard 2D rotation matrices.
    """
    x1 = x[..., : x.shape[-1] // 2]      # First half: [..., :d/2]  // First half: [a, b]
    x2 = x[..., x.shape[-1] // 2 :]      # Second half: [..., d/2:] // # Second half: [c, d]
    return torch.cat((-x2, x1), dim=-1)  # Concatenate [-x2, x1] // # Result: [-c, -d, a, b]


def apply_rotary_pos_emb(q, k, cos, sin, position_ids=None, unsqueeze_dim=1):
    """
    Apply rotary position embedding to query and key tensors so that q_emb and k_emb have relative position information encoded.

    Purpose:
    RoPE encodes position information by rotating query/key vectors
    based on their position. This allows the model to understand relative positions.

    Personal Understanding: just a helper function that pre-rotates the q and k vectors so that when we compute attention scores later (q @ k.T),
    where q and k automatically have relative position information encoded.

    Args:
        q, k: Query and key tensors of shape (batch, heads, seq_len, head_dim)
        cos, sin: Cosine and sine values for rotation
        position_ids: Position indices (unused in this implementation)
        unsqueeze_dim: Which dimension to add for broadcasting

    Returns:
        Rotated query and key tensors with position information encoded

    Our Goal:
        # ============================================================================
        # ROTARY POSITION EMBEDDING (RoPE) APPLICATION
        # ============================================================================
        #(Note: θ is frequency, which is calculated in the RotaryEmbedder class)

        THE FUNDAMENTAL 2D ROTATION MATRIX:
        RoPE is based on this basic rotation matrix R_θ:

            R_θ^(i) = ( cos(mθ_i)  -sin(mθ_i) )
                      ( sin(mθ_i)   cos(mθ_i) )

        where:
        - θ_i is a predefined frequency for feature pair i
        - m is the position index
        - This rotates 2D coordinates by angle mθ_i

        Each feature pair (x_m^(i), x_m^(i+d/2)) is treated as 2D coordinates and multiplied by the rotation matrix R_θ^(i):

        ( x_m^(i)'    )     R_θ^(i) ( x_m^(i)     )     ( cos(mθ_i)  -sin(mθ_i) ) ( x_m^(i)     )
        ( x_m^(i+d/2)') =           ( x_m^(i+d/2) )  =  ( sin(mθ_i)   cos(mθ_i) ) ( x_m^(i+d/2) )

        This matrix multiplication gives us the RoPE formula below.


        THE OFFICIAL RoPE FORMULA (from the paper):
        For a d-dimensional vector x_m at position m, the rotation is:

            ⎛ x_m^(i) cos(mθ_i) - x_m^(i+d/2) sin(mθ_i) ⎞
            ⎜                                              ⎟
            ⎝ x_m^(i+d/2) cos(mθ_i) + x_m^(i) sin(mθ_i) ⎠

        for i ∈ 1, 2, ..., d/2

        WHERE:
        - x_m^(i) = element i of vector at position m
        - x_m^(i+d/2) = element i+d/2 of vector at position m
        - θ_i = 1/10000^(2i/d) (frequency for dimension pair i)
        - m = position index

        EXPLANATION OF THE PAIRING PATTERN:
        This formula pairs dimension i with dimension (i + d/2), NOT with (i + 1)!

        # For 6D vector [a, b, c, d, e, f] (d=6, so d/2=3):
        # - i=1: pairs element 1 (a) with element 1+3=4 (d) → a*cos - d*sin, d*cos + a*sin
        # - i=2: pairs element 2 (b) with element 2+3=5 (e) → b*cos - e*sin, e*cos + b*sin
        # - i=3: pairs element 3 (c) with element 3+3=6 (f) → c*cos - f*sin, f*cos + c*sin

        EXAMPLE:
        ---
        For 2D, our desired output should be:
        [x'] = [cos(θ) -sin(θ)] [x]
        [y']   [sin(θ)  cos(θ)] [y]

        x' = x*cos(θ) - y*sin(θ)
        y' = x*sin(θ) + y*cos(θ)
        ---

        For 6D, our desired output should be:
        Apply the RoPE formula pairing first half with second half:

        Using RoPE pairing: dimension i with dimension (i + d/2)
        d=6, so d/2=3

        Pair 1: element 0 (a) with element 3 (d) using frequency θ₀:
        a' = a*cos(θ₀) - d*sin(θ₀)
        d' = d*cos(θ₀) + a*sin(θ₀)

        Pair 2: element 1 (b) with element 4 (e) using frequency θ₁:
        b' = b*cos(θ₁) - e*sin(θ₁)
        e' = e*cos(θ₁) + b*sin(θ₁)

        Pair 3: element 2 (c) with element 5 (f) using frequency θ₂:
        c' = c*cos(θ₂) - f*sin(θ₂)
        f' = f*cos(θ₂) + c*sin(θ₂)

        For input vector [a, b, c, d, e, f], the desired output is:
        [a*cos(θ₀) - d*sin(θ₀),   # Pair 1, first element   ; Position 0
         b*cos(θ₁) - e*sin(θ₁),   # Pair 2, first element   ; Position 1
         c*cos(θ₂) - f*sin(θ₂),   # Pair 3, first element   ; Position 2
         d*cos(θ₀) + a*sin(θ₀),   # Pair 1, second element  ; Position 3
         e*cos(θ₁) + b*sin(θ₁),   # Pair 2, second element  ; Position 4
         f*cos(θ₂) + c*sin(θ₂)]   # Pair 3, second element  ; Position 5

        # HOW RoPE'S VECTORIZED FORMULA ACHIEVES THIS:
        # The formula: q*cos + rotate_half(q)*sin with specially arranged cos/sin patterns
        # produces the SAME result as the individual 2D rotations above, but efficiently!
        #
        # ============================================================================



    DETAILED EXAMPLE of HOW RoPE'S VECTORIZED FORMULA ACHIEVES THIS: (6D head_dim):
    Input vector: q = [a, b, c, d, e, f]

    Step 1 - rotate_half rearrangement:
        rotate_half([a, b, c, d, e, f]) = [-d, -e, -f, a, b, c]
        Purpose: Creates cross-terms between first half [a,b,c] and second half [d,e,f]

    Step 2 - cos/sin pattern (frequency pairs):
        cos = [cos₀, cos₀, cos₁, cos₁, cos₂, cos₂]  # Each frequency repeated twice
        sin = [sin₀, sin₀, sin₁, sin₁, sin₂, sin₂]  # Each frequency repeated twice
        Purpose: Different frequencies encode different "scales" of relative position

    Step 3 - Apply vectorized rotation formula:
        Term 1: q * cos = [a*cos₀, b*cos₀, c*cos₁, d*cos₁, e*cos₂, f*cos₂]
        Term 2: rotate_half(q) * sin = [-d*sin₀, -e*sin₀, -f*sin₁, a*sin₁, b*sin₂, c*sin₂]

        Final result = Term 1 + Term 2:
        Position 0: a*cos₀ - d*sin₀  ← First half mixed with second half
        Position 1: b*cos₀ - e*sin₀  ← First half mixed with second half
        Position 2: c*cos₁ - f*sin₁  ← First half mixed with second half
        Position 3: d*cos₁ + a*sin₁  ← Second half mixed with first half
        Position 4: e*cos₂ + b*sin₂  ← Second half mixed with first half
        Position 5: f*cos₂ + c*sin₂  ← Second half mixed with first half

    """
    cos = cos.unsqueeze(unsqueeze_dim)              # Add dimension for broadcasting   
    sin = sin.unsqueeze(unsqueeze_dim)              # Add dimension for broadcasting
    q_embed = (q * cos) + (rotate_half(q) * sin)   # Apply rotation to queries
    k_embed = (k * cos) + (rotate_half(k) * sin)   # Apply rotation to keys
    return q_embed, k_embed

def repeat_kv(hidden_states, n_rep):
    """
    Repeat key-value heads for grouped-query attention.

    Purpose: In grouped-query attention, we have fewer key-value heads than query heads.
    This function repeats each k,v head n_rep times to match the number of query heads.

    WHY WE NEED THIS:
    Later in the attention computation, we need to perform:
        attn_weights = torch.matmul(q_states, k_states.transpose(2, 3))
        attn_output = torch.matmul(attn_weights, v_states)

    These operations require q_states, k_states, and v_states to have the SAME number of heads:
        q_states.shape = (batch, 9, seq_len, head_dim)  # 9 query heads
        k_states.shape = (batch, 9, seq_len, head_dim)  # Must be 9 to match
        v_states.shape = (batch, 9, seq_len, head_dim)  # Must be 9 to match

    Without this function, we'd have:
        k_states.shape = (batch, 3, seq_len, head_dim)  # Only 3 heads - MISMATCH!
        v_states.shape = (batch, 3, seq_len, head_dim)  # Only 3 heads - MISMATCH!

    And torch.matmul would fail due to incompatible dimensions.

    USAGE IN CODE:
    This function is called in RopeAttention.forward():
        __kv_groups = self.num_heads / self.kv_heads  # 9/3 = 3
        k_states = repeat_kv(k_states, __kv_groups)  # Expand 3 heads to 9
        v_states = repeat_kv(v_states, __kv_groups)  # Expand 3 heads to 9
        # Now attention computation works with matching shapes


    Args:
        hidden_states: K or V tensor of shape (batch, kv_heads, seq_len, head_dim)
        n_rep: Number of times to repeat each head (num_heads // kv_heads)

    Returns:
        Expanded tensor of shape (batch, num_heads, seq_len, head_dim)

    Example: If we have 3 kv_heads and 9 query_heads, n_rep=3
    Each kv_head gets repeated 3 times to match 9 query_heads
    """
    batch, num_key_value_heads, slen, head_dim = hidden_states.shape
    # Insert new dimension and expand
    hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)
    # Reshape to combine repeated heads
    return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)

class RotaryEmbedder(nn.Module):
    """
    PURPOSE: Pre-compute arguments cos and sin used later in apply_rotary_pos_emb(q, k, cos, sin, ...)

    USAGE PATTERN:
        # Step 1: Calculate rotation angles (this class)
        rope = RotaryEmbedder(dim=64, base=10000)
        cos, sin = rope(some_tensor)  # Get rotation angles for all positions

        # Step 2: Apply rotations to actual vectors (apply_rotary_pos_emb)
        q_rotated, k_rotated = apply_rotary_pos_emb(q, k, cos, sin)

    Thinking Process: For the RoPE formula, we need to pre-compute the cos(mθ_i) and sin(mθ_i) values needed

    THE OFFICIAL RoPE FORMULA (from the paper):
    For a d-dimensional vector x_m at position m, the rotation is:

        ⎛ x_m^(i) cos(mθ_i) - x_m^(i+d/2) sin(mθ_i) ⎞
        ⎜                                              ⎟
        ⎝ x_m^(i+d/2) cos(mθ_i) + x_m^(i) sin(mθ_i) ⎠


    THIS CLASS'S ROLE:
    Compute and organize the cos(mθ_i) and sin(mθ_i) coefficients so that
    apply_rotary_pos_emb can efficiently apply the RoPE formula to actual vectors.

    OUR PROCESS:
    1. Calculate frequencies θ_i for each dimension pair
    2. For each position m, compute angles mθ_i for all frequencies
    3. Arrange angles in RoPE pairing pattern: [mθ₀, mθ₀, mθ₁, mθ₁, ...] for each position m
    4. Return cos([mθ₀, mθ₀, mθ₁, mθ₁, ...]) and sin([mθ₀, mθ₀, mθ₁, mθ₁, ...])

    OUTPUT EXPECTATION (I guess it's because apply_rotary_pos_emb needs arguments cos and sin to be in this shape):
    cos, sin tensors of shape (batch, seq_len, head_dim) where:
    - Each position m has its own row of cos/sin values
    - Each row contains the frequency pattern: [θ₀, θ₀, θ₁, θ₁, ..., θ_{d/2-1}, θ_{d/2-1}]
    - Ready to be used in apply_rotary_pos_emb for the RoPE formula
    """
    def __init__(self, dim, base):
        """
        Initialize RoPE with frequency calculation.

        GOAL: Pre-compute the frequencies θ_i needed for the RoPE formula.

        Args:
            dim: Dimension of each attention head (head_dim, e.g., 64)
            base: Base for frequency calculation (usually 10000)

        """
        super().__init__()
        # Calculate frequencies for different dimensions
        # freq[i] = 1 / (base^(2i/dim)) for i in [0, 2, 4, ..., dim-2]
        # e.g. For dimension of 64, we have 32 frequency pairs (64/2 = 32)
        # Formula is on the markdown: $$\theta_i = 10000^{-2i/d_{model}}$$
        self.freq = 1/(base ** (torch.arange(0, dim, 2, dtype=torch.int64).float()/dim))

    @torch.no_grad()
    def forward(self,x):
        """
        Generate cosine and sine values for RoPE formula.

        GOAL: Compute cos(mθ_i) and sin(mθ_i) for all positions m and frequencies θ_i,
        arranged in the pattern needed by apply_rotary_pos_emb.

        Args:
            x: Input tensor (we only use x.shape[-2] to determine seq_len)

        Returns:
            cos, sin: Tensors of shape (batch, seq_len, head_dim) containing
                     rotation coefficients for the RoPE formula

        PROCESS:
        1. Extract positions: m = [0, 1, 2, ..., seq_len-1]
        2. Compute angles: mθ_i for each (position, frequency) pair
        3. Arrange for RoPE pairing: duplicate each frequency for the pairing pattern
        4. Return cos() and sin() of the arranged angles
        """

        pos = torch.arange(x.shape[-2],dtype=torch.long) # STEP 1: Get position indices m = [0, 1, 2, ..., seq_len-1], Shape: (seq_len,) = e.g. (10,)

        # STEP 2: Because we are expecting each position and each indice at the position to rotate an angle. In other words,
        # We are expecting each feature at each position to rotate an angle, so we need an angle for each feature at each position.
        # angles Shape: (seq_len, dim/2) = (10, 32)
        # angles[m,i] = pos[m] * freq[i] = mθ_i (exactly what RoPE formula needs!)

        ### BUG FIX #1 ###
        # Current (INCORRECT):
        # angles = torch.einsum('f,p->fp', self.freq, pos.float()).unsqueeze(dim=0)
        # Should be (CORRECT):
        angles = torch.einsum('f,p->pf', self.freq, pos.float()).unsqueeze(dim=0)
        # Issue: # 'f,p->fp' produces (freq_dim, seq_len) but we need (seq_len, freq_dim)
        # This causes dimension mismatch when broadcasting with q_states in apply_rotary_pos_emb.
        # apply_rotary_pos_emb expects cos/sin with shape (batch, seq_len, head_dim)
        # 'f,p->pf' correctly produces (seq_len, freq_dim) for proper broadcasting
        ######

        # STEP 3: Duplicate angles for RoPE's pairing pattern
        # RoPE pairs dimension i with dimension (i+d/2), so each frequency appears twice
        # Pattern: [θ₀, θ₀, θ₁, θ₁, θ₂, θ₂, ..., θ_{d/2-1}, θ_{d/2-1}]
        # Shape: (1, seq_len, head_dim) = (1, 10, 64)
        # emb[0,m,:] = [mθ₀, mθ₀, mθ₁, mθ₁, ..., mθ₃₁, mθ₃₁] for position m
        emb = torch.cat((angles, angles), dim=-1)
        return emb.cos(), emb.sin()  # STEP 4: Return cos(mθ_i) and sin(mθ_i) as needed by RoPE formula

# ============================================================================
# MLP (FEEDFORWARD) BLOCK
# ============================================================================
#
# ARCHITECTURE COMPARISON:
# Standard MLP (2 layers):
#   x -> Linear -> Activation -> Linear -> output
#
# SmolLM MLP (3 layers - SwiGLU pattern):
#   x -> Linear (gate) -> SiLU activation
#   x -> Linear (up)   -> (no activation)
#   gate_output * up_output -> Linear (down) -> output
#
# WHY 3 LAYERS?
# SwiGLU activation pattern requires:
# 1. Gate pathway: transforms input and applies activation
# 2. Up pathway: transforms input without activation
# 3. Element-wise multiplication of the two pathways
# 4. Down projection back to original dimension
# ============================================================================

class MLP(nn.Module):
    """
    Multi-Layer Perceptron with SwiGLU activation (3 linear layers).

    Purpose: This is the feedforward component of each transformer layer.
    SmolLM uses SwiGLU activation which requires 3 linear layers instead of 2.

    SwiGLU Formula: W_down(SiLU(W_gate(x)) * W_up(x))

    PATHWAY BREAKDOWN:
    1. Gate pathway: W_gate(x) -> SiLU -> gate_output
    2. Up pathway:   W_up(x) -> up_output (no activation)
    3. Combine:      gate_output * up_output (element-wise multiplication)
    4. Project down: W_down(combined) -> final_output

    This pattern has been shown to improve model performance compared to standard MLP.
    """

    def __init__(self, hidden_size, intermediate_size):
        """
        Initialize the 3-layer SwiGLU MLP.

        Args:
            hidden_size: Input/output dimension (576 for SmolLM-135M)
            intermediate_size: Hidden dimension (1536 for SmolLM-135M, ~2.67x expansion)
        """
        super().__init__()
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size

        # The three linear transformations for SwiGLU
        self.W_gate = nn.Linear(self.hidden_size, self.intermediate_size, bias=False)  # Gate projection
        self.W_up = nn.Linear(self.hidden_size, self.intermediate_size, bias=False)    # Up projection
        self.W_down = nn.Linear(self.intermediate_size, self.hidden_size, bias=False) # Down projection

        self.act_fn = torch.nn.modules.activation.SiLU()  # SiLU activation (x * sigmoid(x))

    def forward(self, x):
        """
        Forward pass through SwiGLU MLP.

        Process:
        1. Gate pathway: SiLU(W_gate(x))
        2. Up pathway:   W_up(x)
        3. Combine:      gate * up (element-wise)
        4. Project:      W_down(combined)

        Args:
            x: Input tensor of shape (..., hidden_size)

        Returns:
            Output tensor of shape (..., hidden_size)
        """
        # Apply SwiGLU formula: W_down(SiLU(W_gate(x)) * W_up(x))
        gate_output = self.act_fn(self.W_gate(x))    # Gate pathway with activation
        up_output = self.W_up(x)                     # Up pathway without activation
        combined = gate_output * up_output           # Element-wise multiplication
        down_proj = self.W_down(combined)            # Project back to original dimension
        return down_proj






# RMSNorm (Root Mean Square Normalization) Mathematics

RMSNorm is a simpler alternative to LayerNorm that normalizes activations using only the root mean square, without centering (no mean subtraction).

## Core Formula

For an input vector $x = (x_1, x_2, ..., x_d)$ of dimension $d$, RMSNorm computes:

$$y_i = \frac{x_i}{\text{RMS}(x)} \cdot g_i$$

where:
- $\text{RMS}(x) = \sqrt{\frac{1}{d} \sum_{j=1}^{d} x_j^2}$ is the root mean square

- $g_i$ is the learnable **weight** parameter (gain/scale) for dimension $i$

## Step-by-Step Breakdown

1. **Compute RMS**: $$\text{RMS}(x) = \sqrt{\frac{x_1^2 + x_2^2 + ... + x_d^2}{d}}$$

2. **Normalize**: $$\hat{x}_i = \frac{x_i}{\text{RMS}(x)}$$

3. **Scale with learnable weights**: $$y_i = \hat{x}_i \cdot g_i = \frac{x_i}{\text{RMS}(x)} \cdot g_i$$

In [ ]:
class RMSNorm(nn.Module):
    """
    Root Mean Square Layer Normalization.

    Purpose: Alternative to LayerNorm that's computationally simpler.
    Instead of subtracting mean and dividing by std, RMSNorm just divides by RMS.

    COMPARISON:
    LayerNorm: y = (x - mean(x)) / sqrt(var(x) + eps) * weight + bias
    RMSNorm:   y = x / sqrt(mean(x²) + eps) * weight

    Benefits:
    - No mean subtraction (saves computation)
    - No bias term needed
    - Achieves similar normalization effects
    """

    def __init__(self, hidden_size, eps=1e-6):
        """
        Initialize RMSNorm layer.

        Args:
            hidden_size: Size of the last dimension
            eps: Small epsilon for numerical stability (prevents division by zero)
        """
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))    # Learnable scale parameter, the "weight" in the formula
        self.variance_epsilon = eps                            # Numerical stability

    def forward(self, hidden_states):
        """
        Apply RMS normalization.

        Args:
            hidden_states: Input tensor of shape (..., hidden_size)

        Returns:
            Normalized tensor scaled by learned weight, the "y" in the formula
        """
        # STEP 1: Calculate mean of squares (this IS the variance for zero-mean)
        # For input tensor x of shape (..., hidden_size):
        # x²  = hidden_states.pow(2)                    # Square each element: [x₁², x₂², x₃², ...]
        # variance = x².mean(-1, keepdim=True)          # Average across last dimension: (x₁² + x₂² + ... + xₙ²) / n
        # Why keepdim=True? Keeps the dimension for broadcasting: (batch, seq, hidden_size) → (batch, seq, 1)
        variance = hidden_states.pow(2).mean(-1, keepdim=True)  # Mean of squares: E[x²] = (x₁² + x₂² + ... + xₙ²) / n

        # STEP 2: Divide by RMS (Root Mean Square)
        ######
        ### BUG FIX 2 ###
        # Current (INCORRECT):
        # hidden_states = hidden_states * torch.sqrt(variance + self.variance_epsilon)
        # Should be (CORRECT):
        hidden_states = hidden_states / torch.sqrt(variance + self.variance_epsilon) # x / √(E[x²] + ε)
        # Issue: Incorrect operation for RMSNorm - using multiplication instead of division.
        ######

        # STEP 3: Apply learnable scaling
        return self.weight * hidden_states # y = x_i^hat * g_i




# ============================================================================
# MULTI-HEAD ATTENTION WITH ROPE
# ============================================================================

class RopeAttention(nn.Module):
    """
    Multi-head attention with RoPE position encoding and grouped-query attention.

    PURPOSE: Computes context vectors that capture relevant information from the entire
    sequence for each position. This is the core mechanism that allows the transformer
    to "attend" to different parts of the input when processing each token.

    OUTPUT: Context vectors (attended representations), NOT attention scores.
    The output contains a weighted combination of value vectors based on attention patterns.

    DEPENDENCIES (classes this interacts with):
    - RotaryEmbedder: Provides cos/sin rotation coefficients for position encoding
    - apply_rotary_pos_emb: Applies RoPE rotations to query and key vectors
    - repeat_kv: Expands key-value heads for grouped-query attention

    USAGE IN LATER COMPONENTS:
    - Used by LlamaDecoder: This attention output gets added to residual connections
    - Feeds into MLP: The context vectors become input to the feedforward network
    - Model composition: Part of the transformer layer that processes sequences

    ARCHITECTURE COMPONENTS:
    1. Multi-head attention: 9 query heads for diverse attention patterns
    2. Grouped-query attention: 3 key-value heads shared across query heads (memory efficient)
    3. RoPE position encoding: Relative position information through rotation
    4. Causal masking: Prevents attention to future positions (autoregressive)

    ATTENTION FLOW:
    Input → Q,K,V projections → Reshape for heads → Apply RoPE →
    Repeat K,V heads → Compute attention scores → Apply to values →
    Reshape & project → Output context vectors
    """

    def __init__(self, config):
        """
        Initialize attention layer.

        Args:
            config: Configuration object with model hyperparameters
                - hidden_size: 576 (input/output dimension)
                - num_heads: 9 (query heads)
                - kv_heads: 3 (key-value heads for grouped attention)
        """
        super().__init__()
        self.hidden_size = config.hidden_size                    # 576 (input/output dimension)
        self.num_heads = config.num_heads                        # 9 query heads
        self.head_dim = config.hidden_size // self.num_heads     # 64 dimensions per head (576/9)
        self.kv_heads = config.kv_heads                          # 3 key-value heads (grouped attention)
        self.rope_theta = 10000.0                               # RoPE base frequency

        # Linear projections for Q, K, V (note different sizes for grouped attention)
        self.W_query = nn.Linear(config.hidden_size, self.num_heads * self.head_dim, bias=False)  # 576 → 576 (9*64)
        self.W_key = nn.Linear(config.hidden_size, self.kv_heads * self.head_dim, bias=False)     # 576 → 192 (3*64)
        self.W_value = nn.Linear(config.hidden_size, self.kv_heads * self.head_dim, bias=False)   # 576 → 192 (3*64)
        self.W_output = nn.Linear(config.hidden_size, config.hidden_size, bias=False)             # 576 → 576 (final projection)

        # RoPE embedder for position encoding
        self.rotary_emb = RotaryEmbedder(base=self.rope_theta,
                                         dim=config.hidden_size//self.num_heads)  # dim=64 (head_dim)

    def forward(self, hidden_states: torch.Tensor, attention_mask=None):
        """
        Forward pass through attention mechanism.

        Args:
            hidden_states: Input tensor of shape (batch, seq_len, hidden_size)
            attention_mask: Mask to prevent attention to certain positions

        Returns:
            Attention output of shape (batch, seq_len, hidden_size)
        """
        b, q, _ = hidden_states.size()  # batch_size, seq_len, hidden_size
        # Suppose hidden_states.shape = (batch, seq_len, hidden_size) = (1, 10, 576) for easier later illustration

        # STEP 1: Project input to Q, K, V
        q_states = self.W_query(hidden_states)    # (batch, seq_len, 576) → (batch, seq_len, 576)
        # Matrix operation: hidden_states @ W_query.weight.T + bias
        # Dimensions: (1, 10, 576) @ (576, 576).T + bias
        # Calculation: (1, 10, 576) @ (576, 576) → (1, 10, 576)
        # Result: (batch, seq_len, 576) → (batch, seq_len, 576)
        # In other words(raw shape):
        # Dimensions: (batch, seq_len, hidden_size) @ (num_heads * head_dim, hidden_size).T + bias
        # Calculation: (batch, seq_len, hidden_size) @ (hidden_size, num_heads * head_dim) → (batch, seq_len, num_heads * head_dim)
        # Result: (batch, seq_len, hidden_size) → (batch, seq_len, hidden_size)
        k_states = self.W_key(hidden_states)      # (batch, seq_len, 576) → (batch, seq_len, 192)
        # Matrix operation: hidden_states @ W_key.weight.T + bias
        # Dimensions: (1, 10, 576) @ (192, 576).T + bias
        # Calculation: (1, 10, 576) @ (576, 192) → (1, 10, 192)
        # Result: (batch, seq_len, 576) → (batch, seq_len, 192)
        v_states = self.W_value(hidden_states)    # (batch, seq_len, 576) → (batch, seq_len, 192)
        # Matrix operation: hidden_states @ W_value.weight.T + bias; the following is the same as k_states

        # STEP 2: Reshape for multi-head attention
        q_states = q_states.view(b, q, self.num_heads, self.head_dim).transpose(1, 2)  # (batch, 9, seq_len, 64)
        k_states = k_states.view(b, q, self.kv_heads, self.head_dim).transpose(1, 2)   # (batch, 3, seq_len, 64)
        v_states = v_states.view(b, q, self.kv_heads, self.head_dim).transpose(1, 2)   # (batch, 3, seq_len, 64)

        # STEP 3: Apply RoPE position encoding for relative position information
        ### BUG FIX #10 ###
        # Current (INCORRECT):
        # cos, sin = self.rotary_emb(v_states)
        # Should be (CORRECT):
        cos, sin = self.rotary_emb(q_states)   # Get rotation coefficients
        # Issue: Using v_states (3 heads) to generate rotary embeddings that are applied
        # to q_states (9 heads) and k_states (3 heads) causes shape mismatch and incorrect
        # position encoding.
        q_states, k_states = apply_rotary_pos_emb(q_states, k_states, cos, sin)  # Apply rotations to get q_states and k_states with relative position information encoded

        # STEP 4: Expand K,V heads to match Q heads (grouped-query attention)
        ### BUG FIX #3 ###
        # Current (INCORRECT):
        # __kv_groups = self.num_heads / self.kv_heads
        # Should be (CORRECT):
        __kv_groups = self.num_heads // self.kv_heads # 9/3 = 3 (should be integer division!)
        # Issue: Float division operator (/) returns float 3.0, but expand() requires integer.
        # Integer division (//) returns int 3, which expand() can accept.
        ######
        k_states = repeat_kv(k_states, __kv_groups)              # (batch, 3, seq_len, 64) → (batch, 9, seq_len, 64)
        v_states = repeat_kv(v_states, __kv_groups)              # (batch, 3, seq_len, 64) → (batch, 9, seq_len, 64)

        # STEP 5: Compute attention scores (Q·K^T / √d_k) where d_K = dimension of the key vectors
        # q_states.shape = (1, 9, 10, 64)  # (batch, num_heads, seq_len, head_dim)
        # k_states.shape = (1, 9, 10, 64)  # (batch, num_heads, seq_len, head_dim) - after repeat_kv
        # k_states.transpose(2, 3):
        # After transpose: (1, 9, 64, 10) - (batch, num_heads, head_dim, seq_len)
        # torch.matmul(q_states, k_states.transpose(2, 3))
        # # Dimensions: (1, 9, 10, 64) @ (1, 9, 64, 10)
        # # Matrix math: For each head, (10, 64) @ (64, 10) → (10, 10)          (actual head calculation)
        # # Result: (1, 9, 10, 10) - (batch, num_heads, seq_len, seq_len)
        # #What this represents: For each head, each position (row) attends to every position (column).
        # Next step: scaling (the / √d_k step)
        # Should be: / math.sqrt(self.head_dim) = / math.sqrt(64) = / 8
        # Why 64 or head_dim? Answer: Understand what gets multiplied in the attention calculation (actual head calculation above):
        #   q_head = q_states[0, head_i, :, :]  # Shape: (seq_len, head_dim) = (10, 64)
        #   k_head = k_states[0, head_i, :, :]  # Shape: (seq_len, head_dim) = (10, 64)
        #   # The actual dot product computation:
        #   attention_scores = q_head @ k_head.T  # (10, 64) @ (64, 10) → (10, 10)

        ######
        ### BUG FIX #4 ###
        # Current (INCORRECT):
        # attn_weights = torch.matmul(q_states, k_states.transpose(2, 3)) / math.sqrt(self.hidden_size)
        # Should be (CORRECT):
        attn_weights = torch.matmul(q_states, k_states.transpose(2, 3)) / math.sqrt(self.head_dim)
        # Issue: Wrong scaling factor for attention scores.
        # Should scale by √head_dim (64) not √hidden_size (576) because actual dot products
        # are between head_dim-sized vectors (10, 64) @ (64, 10) for each head individually
        ######



        # attention mask here is + not * because:
        #   # Attention mask contains:
        #   # 0 for positions we CAN attend to
        #   # -∞ for positions we CAN'T attend to
        #   # So when we add the mask, after softmax the -∞ will be 0 and the 0 will be 1
        attn_weights = attn_weights + attention_mask               # Apply causal mask (prevent future attention)
        attn_weights = nn.functional.softmax(attn_weights, dim=-1) # Convert to probabilities

        ######
        ### BUG FIX #5 ###
        # Current (INCORRECT):
        # attn_weights = nn.functional.dropout(attn_weights)
        # Should be (CORRECT):
        attn_weights = nn.functional.dropout(attn_weights, p = 0.0, training=self.training)
        # Issue: Missing training parameter for dropout.
        # Dropout should only be applied during training, not inference.
        # training=self.training ensures dropout is disabled during eval() mode
        # Also, Default dropout probability is 0.5, which can cause issues even in eval mode (or could change the output of the test model).
        # We should set it to dropout probability to 0.0 to pass the test.
        ######

        # STEP 6: Apply attention weights to values
        attn_output = torch.matmul(attn_weights, v_states)         # (batch, 9, seq_len, 64)

        # STEP 7: Reshape back to original format
        attn_output = attn_output.transpose(1, 2).contiguous()     # (batch, seq_len, 9, 64)
        attn_output = attn_output.reshape(b, q, -1)                # (batch, seq_len, 576)


        ######
        ### BUG FIX #6 ###
        # Current (INCORRECT):
        # return attn_output # (MISSING OUTPUT PROJECTION!)
        # Should be (CORRECT):
        attn_output = self.W_output(attn_output)
        return attn_output
        # Issue: Missing output projection layer.
        # Multi-head attention should apply final linear transformation W_output
        # to combine information from all heads before returning result
        ######


# ============================================================================
# TRANSFORMER DECODER LAYER
# ============================================================================

class LlamaDecoder(nn.Module):

    """
    Single transformer decoder layer with attention and MLP.

    Purpose: This represents one layer of the transformer. SmolLM has 30 of these.
    Each layer implements the standard transformer architecture with pre-normalization.

    ARCHITECTURE PATTERN (Pre-Norm):
    Input → Norm → Attention → Add Residual → Norm → MLP → Add Residual → Output

    COMPONENTS USED:
    - RopeAttention: Multi-head attention with RoPE position encoding
    - MLP: SwiGLU feedforward network (3-layer)
    - RMSNorm: Root mean square normalization (2 instances)

    USAGE IN MODEL:
    - Called by smolModel: Processes hidden states through 30 sequential layers
    - Output feeds into next decoder layer or final normalization
    - Part of autoregressive language modeling pipeline

    RESIDUAL CONNECTIONS:
    Critical for training deep networks - allows gradients to flow directly
    through the network and helps with vanishing gradient problem.

    Architecture Visualization (for better visualization go look at the website:https://docs.nvidia.com/deeplearning/transformer-engine/user-guide/examples/te_llama/tutorial_accelerate_hf_llama_with_te.html:

    Input: hidden_states (batch, seq_len, hidden_size)
        ↓
    ┌─────────────────────────────────────────────────────────┐
    │                  ATTENTION BLOCK                        │
    │                                                         │
    │  residual_1 = hidden_states                            │
    │      ↓                                                 │
    │  pre_attn_rmsnorm(hidden_states)                       │
    │      ↓                                                 │
    │  RopeAttention(normalized_states)                      │
    │      ↓                                                 │
    │  attention_output + residual_1                         │
    │                                                         │
    └─────────────────────────────────────────────────────────┘
        ↓
    ┌─────────────────────────────────────────────────────────┐
    │                    MLP BLOCK                            │
    │                                                         │
    │  residual_2 = attention_output                         │
    │      ↓                                                 │
    │  pre_mlp_rmsnorm(attention_output)                     │
    │      ↓                                                 │
    │  MLP(normalized_states)                                │
    │      ↓                                                 │
    │  mlp_output + residual_2                               │
    │                                                         │
    └─────────────────────────────────────────────────────────┘
        ↓
    Output: final_hidden_states (batch, seq_len, hidden_size)
    """

    def __init__(self, config):
        """
        Initialize decoder layer components.

        Args:
            config: Configuration object containing:
                - hidden_size: Model dimension (576)
                - intermediate_size: MLP hidden dimension (1536)
                - num_heads, kv_heads: Attention configuration
        """
        super().__init__()
        self.self_attn = RopeAttention(config)              # Self-attention with RoPE
        self.mlp = MLP(hidden_size=config.hidden_size,      # SwiGLU feedforward network
                      intermediate_size=config.intermediate_size)
        self.pre_attn_rmsnorm = RMSNorm(config.hidden_size, eps=1e-05)  # Pre-attention normalization
        self.pre_mlp_rmsnorm = RMSNorm(config.hidden_size, eps=1e-05)   # Pre-MLP normalization
        # Note that we are creating two separate RMSNorm objects though they have the same initial values.
        # They are stored in different places in memory, so Pytorch will treat them as different objects. So they have the same values initially but different IDENTITIES.
        # We separate them because their weights are updated differently:
        #
        # # Computation flow:
        # input → pre_attn_rmsnorm → attention → pre_mlp_rmsnorm → mlp → loss
        #             ↑                            ↑
        #         Gets gradients from           Gets gradients from
        #         attention's needs             MLP's needs

        #
        # self.pre_attn_rmsnorm.weight += learning_rate * self.pre_attn_rmsnorm.weight.grad
        # self.pre_mlp_rmsnorm.weight += learning_rate * self.pre_mlp_rmsnorm.weight.grad
        # But self.pre_attn_rmsnorm.weight and self.pre_mlp_rmsnorm.weight are different!
        # # Chain rule for pre_attn_rmsnorm.weight:
        # ∂loss/∂(pre_attn_weight) = ∂loss/∂mlp_out × ∂mlp_out/∂mlp_in × ∂mlp_in/∂attn_out × ∂attn_out/∂attn_in × ∂attn_in/∂(pre_attn_weight)

        # # Chain rule for pre_mlp_rmsnorm.weight:
        # ∂loss/∂(pre_mlp_weight) = ∂loss/∂mlp_out × ∂mlp_out/∂mlp_in × ∂mlp_in/∂(pre_mlp_weight)

        ## Why RMSNorm Layers Get Different Gradients: The Chain Rule Effect
        # Each normalization layer sits in a **different position** in the computation graph, so changes to their parameters affect the final loss through **different pathways**:
        # - **pre_attn_rmsnorm**: Affects loss through → attention → pre_mlp_rmsnorm → mlp → loss
        # - **pre_mlp_rmsnorm**: Affects loss through → mlp → loss (shorter path)

    def forward(self, hidden_states, attention_mask):
        """
        Forward pass through decoder layer.

        TRANSFORMER FLOW (Pre-Norm Pattern):
        1. Normalize → Attention → Add Residual
        2. Normalize → MLP → Add Residual

        Args:
            hidden_states: Input tensor (batch, seq_len, hidden_size)
            attention_mask: Mask for attention (unused - causal mask created internally)

        Returns:
            Tuple containing output hidden states of shape (batch, seq_len, hidden_size)
        """
        # ATTENTION BLOCK
        residual = hidden_states                            # Store original input for residual, shape: (batch, seq_len, hidden_size)
        hidden_states = self.pre_attn_rmsnorm(hidden_states)  # Pre-normalize before attention, shape: (batch, seq_len, hidden_size)

        # Create causal attention mask (prevents attention to future positions)
        # attention_mask.shape[-1] = seq_len
        # torch.full((4, 4), fill_value=float('-inf')): creates a 4x4 matrix with all elements set to -inf
        # so torch.full((attention_mask.shape[-1], attention_mask.shape[-1]), fill_value=float('-inf')) = torch.full((seq_len, seq_len), fill_value=-inf)
        # = create a seq_len x seq_len matrix with all elements set to -inf
        # torch.triu(..., diagonal=1):
        # triu = "triangle upper"
        # diagonal=1 means keep upper triangle ABOVE the main diagonal
        # Everything else becomes 0:
        # Example: (say seq_len = 4)
        # [[0, -∞, -∞, -∞],      # Position 0 can't see positions 1,2,3
        # [0,  0, -∞, -∞],      # Position 1 can't see positions 2,3
        # [0,  0,  0, -∞],      # Position 2 can't see position 3
        # [0,  0,  0,  0]]      # Position 3 can see all positions
        attention_mask = torch.triu(torch.full((attention_mask.shape[-1], attention_mask.shape[-1]),
                                              fill_value=float('-inf')), diagonal=1) # shape: (seq_len, seq_len)

        # Apply self-attention (gets context vectors from sequence)
        hidden_states = self.self_attn(
            hidden_states=hidden_states,
            attention_mask=attention_mask,
        ) # shape: (batch, seq_len, hidden_size)
        hidden_states += residual                           # Add residual connection, shape: (batch, seq_len, hidden_size)


        # MLP BLOCK

        ######
        ### BUG FIX #7 ###
        # Current (INCORRECT):
        # (missing residual = hidden_states)
        # Should be (CORRECT):
        residual = hidden_states                            # Store attention output for residual, shape: (batch, seq_len, hidden_size)
        # Issue: Need to update residual after attention block for proper residual connections.
        # Each block should have its own residual connection: input → normalize → transform → add residual
        # The MLP residual should be the output of attention block, not the original input
        ######

        hidden_states = self.pre_mlp_rmsnorm(hidden_states) # Pre-normalize, shape: (batch, seq_len, hidden_size)
        hidden_states = self.mlp(hidden_states)            # Apply SwiGLU MLP
        # MLP internal shapes:
        #   Input: (1, 10, 576)
        #   W_gate projection: (1, 10, 576) → (1, 10, 1536)
        #   W_up projection: (1, 10, 576) → (1, 10, 1536)
        #   Element-wise multiply: (1, 10, 1536) * (1, 10, 1536) = (1, 10, 1536)
        #   W_down projection: (1, 10, 1536) → (1, 10, 576)
        # hidden_states: (1, 10, 576) - MLP output, back to original size
        hidden_states += residual                           # Add residual,
        # shape = element-wise addition between residual (1,10,576) and hidden_states (1, 10, 576) = final output shape (1,10,576)

        outputs = (hidden_states,)                          # Return as tuple for consistency with HuggingFace API
        return outputs


# ============================================================================
# MAIN MODEL CLASS (CORE TRANSFORMER)
# ============================================================================

class smolModel(nn.Module):
    """
    Main SmolLM model containing embeddings, transformer layers, and final norm.

    Purpose: This is the core language model that processes token sequences
    through multiple transformer layers to create contextualized representations.

    ARCHITECTURE PIPELINE:
    Token IDs → Embeddings → 30x LlamaDecoder → Final Norm → Hidden States

    COMPONENTS:
    - Token embeddings: Convert discrete tokens to continuous vectors
    - 30 transformer layers: Sequential processing through LlamaDecoder layers
    - Final normalization: RMSNorm applied to output representations

    INPUT/OUTPUT:
    - Input: Token IDs (discrete integers representing words/subwords)
    - Output: Contextualized hidden states (continuous vectors with sequence context)

    USAGE IN LARGER MODEL:
    - Called by smolLM: This model's output gets projected to vocabulary space
    - Output feeds into: lm_head for next-token prediction logits
    - Role: Core representation learning component of the language model

    Note: the input is token IDs instead of raw text because raw text can't be processed by neural networks - they only understand numbers!
    Usual Text processing pipeline: raw text → tokenization (happens BEFORE the model) → convert to token IDs (happens BEFORE the model) → THIS model receives token IDs

    Example Usuage:

    # Outside the model (typically in your training/inference code):
    tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M")
    text = "The cat is sleeping"
    inputs = tokenizer(text, return_tensors="pt")  # This creates token_ids

    # Then pass to model:
    model_output = model(inputs['input_ids'])  # Model only sees numbers
    """

    def __init__(self, config):
        """
        Initialize the complete transformer model.

        Args:
            config: Configuration object containing:
                - vocab_size: Size of vocabulary (49152)
                - hidden_size: Model dimension (576)
                - num_hidden_layers: Number of transformer layers (30)
                - Other hyperparameters for LlamaDecoder layers
        """
        super().__init__()

        # Token embedding layer: converts token IDs to dense vectors
        self.embed_tokens = nn.Embedding(num_embeddings=config.vocab_size,      # 49152 possible tokens
                                         embedding_dim=config.hidden_size)      # 576-dimensional embeddings

        # Stack of 30 transformer decoder layers
        self.layers = nn.ModuleList([LlamaDecoder(config) for _ in range(config.num_hidden_layers)])  # 30 identical layers
        # nn.ModuleList is a container that holds multiple PyTorch modules and properly registers them for training.
        # Creates a list of 30 LlamaDecoder instances:
        # self.layers = nn.ModuleList([
        #     LlamaDecoder(config),  # Layer 0
        #     LlamaDecoder(config),  # Layer 1
        #     LlamaDecoder(config),  # Layer 2
        #     ...                    # 30 total layers
        #     LlamaDecoder(config)   # Layer 29
        # ])

        # Final normalization applied to output representations
        self.norm = RMSNorm(config.hidden_size, eps=1e-05)                      # Final RMSNorm layer

    def forward(self, input_ids=None, attention_mask=None):
        """
        Forward pass through the entire transformer model.

        PROCESSING FLOW:
        1. Convert token IDs to embeddings
        2. Process through 30 sequential transformer layers
        3. Apply final normalization
        4. Return contextualized representations

        Args:
            input_ids: Token IDs of shape (batch, seq_len)
            attention_mask: Attention mask (passed to layers but currently unused)

        Returns:
            List containing final hidden states of shape (batch, seq_len, hidden_size)
            Note: Returns list format for compatibility, but contains single tensor
        """
        # STEP 1: Convert token IDs to dense embeddings
        inputs_embeds = self.embed_tokens(input_ids)               # (batch, seq_len) → (batch, seq_len, hidden_size)
        hidden_states = inputs_embeds                              # Initialize hidden states with embeddings
        # We want to initialize hidden states because we want to update it during the forward pass;
        # otherwise, without hidden_states = inputs_embeds, we need to update the input_embeds during the forward pass, which is not what we want.

        # STEP 2: Process through all 30 transformer layers sequentially
        for decoder_layer in self.layers:                         # Each layer processes and refines representations
            layer_outputs = decoder_layer(
                hidden_states,                                     # Input: (batch, seq_len, hidden_size)
                attention_mask=attention_mask,                     # Causal mask created internally by each layer
            )
            hidden_states = layer_outputs[0]                       # Extract hidden states from tuple format
            # After each layer: (batch, seq_len, hidden_size) with richer contextualization

        # STEP 3: Apply final normalization to stabilize output representations
        hidden_states = self.norm(hidden_states)                   # Final RMSNorm: (batch, seq_len, hidden_size)

        # STEP 4: Return contextualized representations
        ######
        ### BUG FIX #9 ###
        # Current (INCORRECT):
        # return [hidden_states]    # Return as list (INCONSISTENT FORMAT!)
        # Should be (CORRECT):
        return (hidden_states,)
        # Issue: Inconsistent return format - should return tuple like LlamaDecoder.
        # List vs tuple can cause subtle differences in PyTorch's internal handling
        # and breaks consistency with other model components.
        ######



# ============================================================================
# LANGUAGE MODEL HEAD (COMPLETE MODEL)
# ============================================================================

class smolLM(nn.Module):
    """
    Complete language model with vocabulary prediction head.

    Purpose: Wraps the base transformer model and adds a linear layer to predict next tokens.
    This is the final, complete language model that can generate text.

    ARCHITECTURE PIPELINE:
    Token IDs → smolModel (30 transformer layers) → Hidden States → lm_head → Logits

    COMPONENTS:
    - smolModel: Core transformer that creates contextualized representations
    - lm_head: Linear projection from hidden space to vocabulary space

    INPUT/OUTPUT:
    - Input: Token IDs (batch, seq_len) - discrete token indices
    - Output: Logits (batch, seq_len, vocab_size) - unnormalized probabilities for next tokens

    USAGE:
    - Training: Use logits with cross-entropy loss for next-token prediction
    - Inference: Apply softmax to logits to get token probabilities
    - Generation: Sample from probability distribution to generate text

    ROLE IN LANGUAGE MODELING:
    This is the complete model that performs autoregressive language modeling -
    predicting the next token given previous tokens in a sequence.
    """

    def __init__(self, config):
        """
        Initialize model and prediction head.

        Args:
            config: Configuration object containing:
                - All parameters needed for smolModel (vocab_size, hidden_size, etc.)
                - hidden_size: Used for lm_head input dimension (576)
                - vocab_size: Used for lm_head output dimension (49152)
        """
        super().__init__()

        # Core transformer model: processes tokens to contextualized representations
        self.model = smolModel(config)                                         # 30-layer transformer

        # Language modeling head: projects hidden states to vocabulary predictions
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)  # 576 → 49152

        ### BUG FIX #11 ###
        # Tie the weights between embeddings and lm_head
        self.lm_head.weight = self.model.embed_tokens.weight
        # Issue: lm_head weights were not in the saved file, causing random initialization.
        # This is detected by the following code in the next chunk:
            # result = __test_model.load_state_dict(torch.load('BareBones_SmolLM-135M.pt'), strict=False)
            # print("Missing keys:", result.missing_keys)
            # print("Unexpected keys:", result.unexpected_keys)
        # Solution: Tie weights with the embedding layer (common in LMs)

    def forward(self, input_ids, attention_mask):
        """
        Forward pass with vocabulary prediction.

        COMPLETE PIPELINE:
        1. Process token IDs through transformer layers (smolModel)
        2. Project final hidden states to vocabulary space (lm_head)
        3. Return logits for next-token prediction

        Args:
            input_ids: Token IDs of shape (batch, seq_len)
            attention_mask: Attention mask (passed through to transformer layers)

        Returns:
            Dictionary with 'logits' key containing vocabulary predictions
            - logits: (batch, seq_len, vocab_size) unnormalized scores for each token
        """
        # STEP 1: Get contextualized representations from transformer
        outputs = self.model(
            input_ids=input_ids,                                              # (batch, seq_len)
            attention_mask=attention_mask,
        )


        # STEP 2: Extract hidden states from model output

        ######
        ### BUG FIX #8 ###
        # Current (INCORRECT):
        # hidden_states = outputs[0].squeeze()   # Remove batch dimension (WRONG!)
        # Should be (CORRECT):
        hidden_states = outputs[0]
        # Should be: hidden_states = outputs[0] to maintain (batch, seq_len, hidden_size)
        # Issue: squeeze() removes batch dimension when batch_size=1, causing shape mismatch.
        # lm_head expects (batch, seq_len, hidden_size) but gets (seq_len, hidden_size).
        # This leads to wrong logits calculation and incorrect token prediction.
        ######


        # STEP 3: Project hidden states to vocabulary space
        logits = self.lm_head(hidden_states)                                  # (seq_len, vocab_size) - WRONG SHAPE!
        # Should be: (batch, seq_len, vocab_size)

        # STEP 4: Ensure proper data type for loss computation
        logits = logits.float()                                               # Ensure float32 precision

        # STEP 5: Return in standard format for language modeling

        # print(f"Final logits.shape: {logits.shape}")
        # print(f"Sample logits[0,-1,:5]: {logits[0,-1,:5] if len(logits.shape)==3 else 'WRONG SHAPE!'}")


        return {'logits': logits}                                             # Dictionary format for compatibility




___
___
___

In [ ]:
# __test_model = smolLM(config)
# result = __test_model.load_state_dict(torch.load('BareBones_SmolLM-135M.pt'), strict=False)
# print("Missing keys (not loaded):", result.missing_keys)
# print("Unexpected keys (in file but not in model):", result.unexpected_keys)
# __test_model.eval()

__test_model = smolLM(config)
__test_model.load_state_dict(torch.load('BareBones_SmolLM-135M.pt'), strict=False)
__test_model.eval()



smolLM(
  (model): smolModel(
    (embed_tokens): Embedding(49152, 576)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoder(
        (self_attn): RopeAttention(
          (W_query): Linear(in_features=576, out_features=576, bias=False)
          (W_key): Linear(in_features=576, out_features=192, bias=False)
          (W_value): Linear(in_features=576, out_features=192, bias=False)
          (W_output): Linear(in_features=576, out_features=576, bias=False)
          (rotary_emb): RotaryEmbedder()
        )
        (mlp): MLP(
          (W_gate): Linear(in_features=576, out_features=1536, bias=False)
          (W_up): Linear(in_features=576, out_features=1536, bias=False)
          (W_down): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLU()
        )
        (pre_attn_rmsnorm): RMSNorm()
        (pre_mlp_rmsnorm): RMSNorm()
      )
    )
    (norm): RMSNorm()
  )
  (lm_head): Linear(in_features=576, out_features=49152, bias=False)
)

# 3. Test

In [ ]:
######################################################################################################################
############################################## DO NOT CHANGE[START] ##################################################
######################################################################################################################

###### TESTING PROMPTS
# Single-Token Quick Test
check_solution(prompt="Given the following film movie by a critic, rate it out of 10. Respond in a single number.\n\nThe movie started off extremely well, but just got worse after that.\nThe storyline was all over the place and everyone acted terribly.\n 10/10 would not recommend! \n\n ",
               num_tokens=1,
               model_A=__reference_model,
               model_B=__test_model)



>>>>>>>>>>>>>>>>>>>>
	Prompt
<<<<<<<<<<<<<<<<<<<<
Given the following film movie by a critic, rate it out of 10. Respond in a single number.

The movie started off extremely well, but just got worse after that.
The storyline was all over the place and everyone acted terribly.
 10/10 would not recommend! 

 


>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
	Model_A Generation
<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
1



>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
	Model_B Generation
<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
1


In [ ]:
# Multi-Token Quick Test
check_solution(prompt="Where is the Nile located?",
               num_tokens=50,
               model_A=__reference_model,
               model_B=__test_model)

######################################################################################################################
############################################### DO NOT CHANGE[END] ###################################################
######################################################################################################################


>>>>>>>>>>>>>>>>>>>>
	Prompt
<<<<<<<<<<<<<<<<<<<<
Where is the Nile located?


>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
	Model_A Generation
<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

The Nile River is located in the Nile Delta in the Nile River Basin, which is a region of Africa. It is the longest river in the world, with a length of 4,330 miles (6,900 km



>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
	Model_B Generation
<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

The Nile River is located in the Nile Delta in the Nile River Basin, which is a region of Africa. It is the longest river in the world, with a length of 4,330 miles (6,900 km
